# DSP Generation Quality Evaluation

**Project:** Vibe-Synth  
**Purpose:** Evaluate the quality of Faust DSP code generated by the LLM pipeline  
**Metrics:** BIBO stability pass rate, compilation success rate, parameter manifest completeness, self-correction rate  

Run all cells top-to-bottom. Requires the backend to be running at `http://localhost:8000`.

In [ ]:
# ---------------------------------------------------------------------------
# Dependencies
# ---------------------------------------------------------------------------
import json
import time
import asyncio
import statistics
import httpx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from IPython.display import display

API_BASE = "http://localhost:8000/api/v1"
print("Dependencies loaded.")

## 1. Test Vibe Set

A curated set of DSP Vibe descriptions covering a range of complexity levels and acoustic archetypes.

In [ ]:
TEST_VIBES = [
    # Simple single-stage
    {"id": "dsp_01", "vibe": "A gentle low-pass filter to warm up a harsh synth", "expected_type": "filter"},
    {"id": "dsp_02", "vibe": "Basic tremolo effect — slow and dreamy", "expected_type": "modulation"},
    {"id": "dsp_03", "vibe": "Hard digital bitcrusher for 8-bit destruction", "expected_type": "distortion"},

    # Medium complexity
    {"id": "dsp_04", "vibe": "Warm tape saturation with gentle tone control", "expected_type": "saturation"},
    {"id": "dsp_05", "vibe": "Small wooden room reverb — intimate and cosy", "expected_type": "reverb"},
    {"id": "dsp_06", "vibe": "Telephone effect — narrow band-pass for voice", "expected_type": "filter"},
    {"id": "dsp_07", "vibe": "Long echo delay with gentle feedback decay", "expected_type": "delay"},

    # Complex multi-stage
    {"id": "dsp_08", "vibe": "Freezing ice cave reverb — vast, cold, and empty", "expected_type": "reverb+filter"},
    {"id": "dsp_09", "vibe": "Lush stereo chorus with warm saturation — thick ensemble sound", "expected_type": "chorus+saturation"},
    {"id": "dsp_10", "vibe": "Gritty bass distortion with heavy compression and tight gate", "expected_type": "distortion+dynamics"},

    # Edge cases
    {"id": "dsp_11", "vibe": "Pass through — do absolutely nothing to the signal", "expected_type": "passthrough"},
    {"id": "dsp_12", "vibe": "Very short reverb tail on a completely dry signal with no feedback whatsoever", "expected_type": "reverb"},
]

print(f"{len(TEST_VIBES)} test Vibes loaded.")

## 2. Run Generation Pipeline

In [ ]:
async def generate_dsp(vibe: str, force_refresh: bool = True) -> dict:
    """Call POST /api/v1/generate/dsp and return the full response dict."""
    async with httpx.AsyncClient(timeout=60.0) as client:
        resp = await client.post(
            f"{API_BASE}/generate/dsp",
            json={
                "prompt":         vibe,
                "input_type":     "audio_stream",
                "sample_rate":    44100,
                "max_parameters": 8,
                "force_refresh":  force_refresh,
            },
        )
        resp.raise_for_status()
        return resp.json()


results = []

for test in TEST_VIBES:
    print(f"Running {test['id']}: {test['vibe'][:60]}…", end=" ", flush=True)
    t0 = time.perf_counter()
    try:
        response = asyncio.run(generate_dsp(test["vibe"]))
        elapsed  = round((time.perf_counter() - t0) * 1000)
        results.append({
            "id":                  test["id"],
            "vibe":                test["vibe"],
            "expected_type":       test["expected_type"],
            "status":              response.get("status"),
            "compile_time_ms":     response.get("compile_time_ms"),
            "cache_hit":           response.get("cache_hit"),
            "param_count":         len(response.get("parameters", [])),
            "corrections":         response.get("self_correction_attempts", 0),
            "faust_len":           len(response.get("faust_code", "")),
            "wasm_module_id":      response.get("wasm_module_id", ""),
            "wall_time_ms":        elapsed,
            "error":               None,
        })
        print(f"OK ({elapsed} ms)")
    except Exception as exc:
        elapsed = round((time.perf_counter() - t0) * 1000)
        results.append({
            "id":            test["id"],
            "vibe":          test["vibe"],
            "expected_type": test["expected_type"],
            "status":        "error",
            "error":         str(exc),
            "wall_time_ms":  elapsed,
        })
        print(f"ERROR: {exc}")

df = pd.DataFrame(results)
print(f"\n{len(df)} tests completed.")

## 3. Summary Statistics

In [ ]:
success_mask  = df["status"].isin(["success", "cache_hit", "self_corrected"])
success_rate  = success_mask.mean() * 100
correction_rate = (df.loc[success_mask, "corrections"] > 0).mean() * 100
avg_compile   = df.loc[success_mask, "compile_time_ms"].mean()
avg_params    = df.loc[success_mask, "param_count"].mean()
zero_params   = (df.loc[success_mask, "param_count"] == 0).sum()

print(f"{'Metric':<35} {'Value':>10}")
print("-" * 47)
print(f"{'Success rate':<35} {success_rate:>9.1f}%")
print(f"{'Self-correction rate (of successes)':<35} {correction_rate:>9.1f}%")
print(f"{'Avg compile time (ms)':<35} {avg_compile:>10.0f}")
print(f"{'Avg parameter count':<35} {avg_params:>10.1f}")
print(f"{'Generations with 0 parameters':<35} {zero_params:>10}")

## 4. Results Table

In [ ]:
display_cols = ["id", "expected_type", "status", "compile_time_ms", "param_count", "corrections", "faust_len"]
display(df[display_cols].style.applymap(
    lambda v: "background-color: #ffcccc" if v == "error" else "",
    subset=["status"],
))

## 5. Compile Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram: compile time
times = df.loc[success_mask, "compile_time_ms"].dropna()
axes[0].hist(times, bins=10, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Compile time (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Compile Time Distribution")
axes[0].axvline(times.mean(), color="red", linestyle="--", label=f"Mean: {times.mean():.0f} ms")
axes[0].legend()

# Bar: parameter count by test
param_df = df.loc[success_mask].set_index("id")["param_count"]
param_df.plot(kind="bar", ax=axes[1], color="teal", edgecolor="white")
axes[1].set_xlabel("Test ID")
axes[1].set_ylabel("Parameter count")
axes[1].set_title("Parameter Count per Generation")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("dsp_quality_plots.png", dpi=150)
plt.show()
print("Plot saved to dsp_quality_plots.png")

## 6. BIBO Stability Check

In [ ]:
import sys
sys.path.insert(0, "..")

from generation.dsp.bibo_checker import BIBOChecker

checker = BIBOChecker()
bibo_results = []

for row in results:
    faust_code = row.get("faust_code", "")
    if not faust_code:
        bibo_results.append({"id": row["id"], "stable": None, "errors": ["no code"], "warnings": []})
        continue
    check = checker.check(faust_code)
    bibo_results.append({
        "id":       row["id"],
        "stable":   check.stable,
        "errors":   check.errors,
        "warnings": check.warnings,
    })

bibo_df = pd.DataFrame(bibo_results)
stable_count   = bibo_df["stable"].sum()
unstable_count = (~bibo_df["stable"].fillna(False)).sum()

print(f"BIBO stable:   {stable_count}/{len(bibo_df)}")
print(f"BIBO unstable: {unstable_count}/{len(bibo_df)}")
display(bibo_df[["id", "stable", "errors", "warnings"]])

## 7. Export Results

In [ ]:
out_path = Path("eval_dsp_results.csv")
df.to_csv(out_path, index=False)
print(f"Results saved to {out_path} ({len(df)} rows).")